<a href="https://colab.research.google.com/github/halimAhtasham/Algorithm_Exercise/blob/main/config.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 01. CONFIGURATION & REPRODUCIBILITY
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import torch

# -----------------------------
# Random seed
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# Make experiments as reproducible as possible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# -----------------------------
# Device
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
Device: cpu


In [2]:
# ============================================================
# 02. REQUIRED LIBRARIES
# ============================================================

!pip -q install shap scikit-learn scipy pandas numpy matplotlib seaborn

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
# ============================================================
# 03. LOAD DATASET
# ============================================================
DATA_PATH = "/content/drive/MyDrive/Datasets/ROSIDS23.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (136681, 84)

Columns:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'B

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.3.4-192.168.3.6-11311-60792-6,192.168.3.6,60792,192.168.3.4,11311,6,07/07/2023 02:10:23 PM,6260,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,192.168.3.4-192.168.3.6-11311-60794-6,192.168.3.6,60794,192.168.3.4,11311,6,07/07/2023 02:10:23 PM,5903,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,192.168.3.4-192.168.3.6-11311-39922-6,192.168.3.6,39922,192.168.3.4,11311,6,07/07/2023 02:10:32 PM,4523,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,192.168.3.4-192.168.3.6-11311-55266-6,192.168.3.6,55266,192.168.3.4,11311,6,07/07/2023 02:11:11 PM,5191,5,5,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
4,192.168.3.6-192.168.3.7-43770-11111-6,192.168.3.7,11111,192.168.3.6,43770,6,07/07/2023 02:10:03 PM,72625778,2200,2212,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [12]:
# ============================================
# STEP 4.1 — Dataset Inspection
# ============================================

data = df.copy()

print("Original shape:", data.shape)

print("\nLabel distribution:")
print(data["Label"].value_counts())

print("\nMissing values:")
print(data.isnull().sum().sum())

print("\nInfinite values:")
print(np.isinf(data.select_dtypes(include=np.number)).sum().sum())

Original shape: (136681, 84)

Label distribution:
Label
Benign       62511
DoS          31000
Subflood     30064
UnauthPub     7817
UnauthSub     5289
Name: count, dtype: int64

Missing values:
272

Infinite values:
278


In [13]:
# ============================================
# STEP 4.2 — Remove Non-Predictive Identifiers
# ============================================

# Columns that should NOT be used as ML features
identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp"
]

data = data.drop(columns=identifier_columns)

print("Shape after removing identifiers:", data.shape)

print("\nRemaining columns:")
print(data.columns.tolist())

Shape after removing identifiers: (136681, 80)

Remaining columns:
['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg

In [14]:
# ============================================
# STEP 4.3 — Convert to Binary Classification
# ============================================

# Benign = 0
# Any other class = Attack = 1

data["Binary_Label"] = (data["Label"] != "Benign").astype(int)

# Remove original multiclass label
data = data.drop(columns=["Label"])

print("Binary label distribution:")
print(data["Binary_Label"].value_counts())

print("\nFinal dataset shape:", data.shape)

Binary label distribution:
Binary_Label
1    74170
0    62511
Name: count, dtype: int64

Final dataset shape: (136681, 80)


In [15]:
# ============================================
# STEP 4.4 — Handle Infinite Values
# ============================================

# Convert +inf and -inf into NaN
data = data.replace([np.inf, -np.inf], np.nan)

print("Total missing values after replacing infinity:")
print(data.isnull().sum().sum())

Total missing values after replacing infinity:
550


In [16]:
missing_columns = data.isnull().sum()

missing_columns = missing_columns[missing_columns > 0]

print("Columns containing missing values:")
print(missing_columns)

Columns containing missing values:
Flow Byts/s    275
Flow Pkts/s    275
dtype: int64


In [17]:
# ============================================
# STEP 5.1 — Separate Features and Target
# ============================================

X = data.drop(columns=["Binary_Label"])
y = data["Binary_Label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (136681, 79)
y shape: (136681,)


In [18]:
# ============================================
# STEP 5.2 — First Train/Test Split
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=SEED
)

print("Training set:", X_train.shape)
print("Temporary set:", X_temp.shape)

Training set: (95676, 79)
Temporary set: (41005, 79)


In [19]:
# ============================================
# STEP 5.3 — Validation/Test Split
# ============================================

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (95676, 79)
Validation: (20502, 79)
Test: (20503, 79)


In [20]:
# ============================================
# STEP 5.4 — Verify Stratification
# ============================================

print("\nTrain attack ratio:", y_train.mean())
print("Validation attack ratio:", y_val.mean())
print("Test attack ratio:", y_test.mean())

print("\nClass counts:")
print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())


Train attack ratio: 0.5426543751829089
Validation attack ratio: 0.5426299873183104
Test attack ratio: 0.5426522947861289

Class counts:
Train:
Binary_Label
1    51919
0    43757
Name: count, dtype: int64

Validation:
Binary_Label
1    11125
0     9377
Name: count, dtype: int64

Test:
Binary_Label
1    11126
0     9377
Name: count, dtype: int64


In [21]:
# ============================================
# STEP 6.1 — Train-Only Missing Value Imputation
# ============================================

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# FIT ONLY on training data
X_train_imputed = imputer.fit_transform(X_train)

# Transform validation and test using training statistics
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("Missing values after imputation:")
print("Train:", np.isnan(X_train_imputed).sum())
print("Validation:", np.isnan(X_val_imputed).sum())
print("Test:", np.isnan(X_test_imputed).sum())

Missing values after imputation:
Train: 0
Validation: 0
Test: 0


In [22]:
# ============================================
# STEP 6.2 — Train-Only Feature Scaling
# ============================================

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# FIT ONLY on training data
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transform validation and test
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaled shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Scaled shapes:
Train: (95676, 79)
Validation: (20502, 79)
Test: (20503, 79)


In [23]:
# ============================================
# STEP 6.3 — Final Preprocessing Check
# ============================================

print("Train min:", X_train_scaled.min())
print("Train max:", X_train_scaled.max())

print("\nValidation min:", X_val_scaled.min())
print("Validation max:", X_val_scaled.max())

print("\nTest min:", X_test_scaled.min())
print("Test max:", X_test_scaled.max())

print("\nAny NaN?")
print("Train:", np.isnan(X_train_scaled).any())
print("Validation:", np.isnan(X_val_scaled).any())
print("Test:", np.isnan(X_test_scaled).any())

Train min: 0.0
Train max: 1.0

Validation min: 0.0
Validation max: 2.0

Test min: 0.0
Test max: 1.1697259168960004

Any NaN?
Train: False
Validation: False
Test: False


In [24]:
# ============================================
# STEP 7.1 — Convert Data to PyTorch
# ============================================

import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation:", X_val_tensor.shape, y_val_tensor.shape)
print("Test:", X_test_tensor.shape, y_test_tensor.shape)

Train: torch.Size([95676, 79]) torch.Size([95676, 1])
Validation: torch.Size([20502, 79]) torch.Size([20502, 1])
Test: torch.Size([20503, 79]) torch.Size([20503, 1])


In [25]:
# ============================================
# STEP 7.2 — Create DataLoaders
# ============================================

BATCH_SIZE = 256

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

Number of training batches: 374
Number of validation batches: 81
Number of test batches: 81


In [26]:
# ============================================
# STEP 7.3 — Neural IDS Model
# ============================================

import torch.nn as nn

class IDSModel(nn.Module):

    def __init__(self, input_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


input_features = X_train_scaled.shape[1]

ids_model = IDSModel(input_features).to(DEVICE)

print(ids_model)

IDSModel(
  (network): Sequential(
    (0): Linear(in_features=79, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [27]:
# ============================================
# STEP 7.4 — Loss and Optimizer
# ============================================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    ids_model.parameters(),
    lr=0.001
)

print("Loss:", criterion)
print("Optimizer:", optimizer)

Loss: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [28]:
# ============================================
# STEP 7.5 — Training Function
# ============================================

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0.0

    for features, labels in loader:

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(features)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * features.size(0)

    average_loss = total_loss / len(loader.dataset)

    return average_loss

In [29]:
# ============================================
# STEP 7.6 — Validation Function
# ============================================

def evaluate_loss(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for features, labels in loader:

            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)

            loss = criterion(logits, labels)

            total_loss += loss.item() * features.size(0)

    average_loss = total_loss / len(loader.dataset)

    return average_loss

In [30]:
# ============================================
# STEP 7.7 — Train IDS
# ============================================

EPOCHS = 20

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        ids_model,
        train_loader,
        criterion,
        optimizer,
        DEVICE
    )

    val_loss = evaluate_loss(
        ids_model,
        val_loader,
        criterion,
        DEVICE
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 01/20 | Train Loss: 0.2956 | Val Loss: 0.2253
Epoch 02/20 | Train Loss: 0.2235 | Val Loss: 0.2190
Epoch 03/20 | Train Loss: 0.2169 | Val Loss: 0.2141
Epoch 04/20 | Train Loss: 0.2133 | Val Loss: 0.2112
Epoch 05/20 | Train Loss: 0.2114 | Val Loss: 0.2106
Epoch 06/20 | Train Loss: 0.2081 | Val Loss: 0.2077
Epoch 07/20 | Train Loss: 0.2057 | Val Loss: 0.2055
Epoch 08/20 | Train Loss: 0.2024 | Val Loss: 0.1949
Epoch 09/20 | Train Loss: 0.1965 | Val Loss: 0.1842
Epoch 10/20 | Train Loss: 0.1726 | Val Loss: 0.1434
Epoch 11/20 | Train Loss: 0.1557 | Val Loss: 0.1289
Epoch 12/20 | Train Loss: 0.1436 | Val Loss: 0.1181
Epoch 13/20 | Train Loss: 0.1327 | Val Loss: 0.1182
Epoch 14/20 | Train Loss: 0.1286 | Val Loss: 0.1142
Epoch 15/20 | Train Loss: 0.1227 | Val Loss: 0.1159
Epoch 16/20 | Train Loss: 0.1206 | Val Loss: 0.1125
Epoch 17/20 | Train Loss: 0.1220 | Val Loss: 0.1126
Epoch 18/20 | Train Loss: 0.1242 | Val Loss: 0.1150
Epoch 19/20 | Train Loss: 0.1230 | Val Loss: 0.1094
Epoch 20/20 

In [31]:
# ============================================
# STEP 8.1 — Generate Test Predictions
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def get_predictions(model, loader, device):

    model.eval()

    all_probabilities = []
    all_labels = []

    with torch.no_grad():

        for features, labels in loader:

            features = features.to(device)

            logits = model(features)

            probabilities = torch.sigmoid(logits)

            all_probabilities.extend(
                probabilities.cpu().numpy().flatten()
            )

            all_labels.extend(
                labels.numpy().flatten()
            )

    return np.array(all_labels), np.array(all_probabilities)


y_test_true, y_test_probability = get_predictions(
    ids_model,
    test_loader,
    DEVICE
)

y_test_pred = (y_test_probability >= 0.5).astype(int)

In [32]:
# ============================================
# STEP 8.2 — Clean IDS Baseline Metrics
# ============================================

accuracy = accuracy_score(y_test_true, y_test_pred)

precision = precision_score(y_test_true, y_test_pred)

recall = recall_score(y_test_true, y_test_pred)

f1 = f1_score(y_test_true, y_test_pred)

roc_auc = roc_auc_score(
    y_test_true,
    y_test_probability
)

pr_auc = average_precision_score(
    y_test_true,
    y_test_probability
)

print("========== CLEAN IDS BASELINE ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

========== CLEAN IDS BASELINE ==========
Accuracy : 0.9685
Precision: 0.9761
Recall   : 0.9656
F1 Score : 0.9708
ROC-AUC  : 0.9885
PR-AUC   : 0.9925


In [33]:
# ============================================
# STEP 8.3 — Confusion Matrix
# ============================================

cm = confusion_matrix(
    y_test_true,
    y_test_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 9114   263]
 [  383 10743]]


In [34]:
# ============================================
# STEP 9.1 — Extract Hidden Representation
# ============================================

class IDSFeatureExtractor(nn.Module):

    def __init__(self, trained_model):
        super().__init__()

        self.feature_network = nn.Sequential(
            *list(trained_model.network.children())[:-1]
        )

    def forward(self, x):
        return self.feature_network(x)


feature_extractor = IDSFeatureExtractor(ids_model).to(DEVICE)

feature_extractor.eval()

print(feature_extractor)

IDSFeatureExtractor(
  (feature_network): Sequential(
    (0): Linear(in_features=79, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
  )
)


In [35]:
# ============================================
# STEP 9.2 — Correctly Classified Benign Data
# ============================================

ids_model.eval()

with torch.no_grad():

    train_logits = ids_model(X_train_tensor.to(DEVICE))

    train_probabilities = torch.sigmoid(train_logits).cpu().numpy().flatten()

train_predictions = (train_probabilities >= 0.5).astype(int)

train_labels = y_train.values

correct_benign_mask = (
    (train_labels == 0) &
    (train_predictions == 0)
)

X_benign_correct = X_train_tensor[correct_benign_mask]

print("Total benign training samples:",
      (train_labels == 0).sum())

print("Correctly classified benign samples:",
      correct_benign_mask.sum())

print("Reconstruction input shape:",
      X_benign_correct.shape)

Total benign training samples: 43757
Correctly classified benign samples: 42682
Reconstruction input shape: torch.Size([42682, 79])


In [36]:
# ============================================
# STEP 9.3 — Extract Benign Representations
# ============================================

with torch.no_grad():

    benign_representations = feature_extractor(
        X_benign_correct.to(DEVICE)
    ).cpu().numpy()

print("Benign representation shape:",
      benign_representations.shape)

Benign representation shape: (42682, 32)


In [37]:
# ============================================
# STEP 10.1 — Reconstruction Dataset
# ============================================

benign_rep_tensor = torch.tensor(
    benign_representations,
    dtype=torch.float32
)

reconstruction_dataset = TensorDataset(
    benign_rep_tensor,
    benign_rep_tensor
)

reconstruction_loader = DataLoader(
    reconstruction_dataset,
    batch_size=256,
    shuffle=True
)

print("Reconstruction samples:", len(reconstruction_dataset))
print("Representation dimension:", benign_rep_tensor.shape[1])

Reconstruction samples: 42682
Representation dimension: 32


In [38]:
# ============================================
# STEP 10.2 — Reconstruction Model
# ============================================

class ReconstructionModel(nn.Module):

    def __init__(self, input_dimension=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dimension, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dimension)
        )

    def forward(self, x):

        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)

        return reconstructed


reconstruction_model = ReconstructionModel(
    input_dimension=32
).to(DEVICE)

print(reconstruction_model)

ReconstructionModel(
  (encoder): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=32, bias=True)
  )
)


In [39]:
# ============================================
# STEP 10.3 — Train Reconstruction Model
# ============================================

reconstruction_optimizer = torch.optim.Adam(
    reconstruction_model.parameters(),
    lr=0.001
)

reconstruction_criterion = nn.MSELoss()

RECONSTRUCTION_EPOCHS = 30

for epoch in range(RECONSTRUCTION_EPOCHS):

    reconstruction_model.train()

    total_loss = 0.0

    for representations, targets in reconstruction_loader:

        representations = representations.to(DEVICE)
        targets = targets.to(DEVICE)

        reconstruction_optimizer.zero_grad()

        reconstructed = reconstruction_model(
            representations
        )

        loss = reconstruction_criterion(
            reconstructed,
            targets
        )

        loss.backward()

        reconstruction_optimizer.step()

        total_loss += (
            loss.item() * representations.size(0)
        )

    average_loss = (
        total_loss / len(reconstruction_loader.dataset)
    )

    if (epoch + 1) % 5 == 0 or epoch == 0:

        print(
            f"Epoch {epoch + 1:02d}/{RECONSTRUCTION_EPOCHS} | "
            f"Reconstruction Loss: {average_loss:.6f}"
        )

Epoch 01/30 | Reconstruction Loss: 2.937767
Epoch 05/30 | Reconstruction Loss: 0.003114
Epoch 10/30 | Reconstruction Loss: 0.002690
Epoch 15/30 | Reconstruction Loss: 0.000754
Epoch 20/30 | Reconstruction Loss: 0.001119
Epoch 25/30 | Reconstruction Loss: 0.002085
Epoch 30/30 | Reconstruction Loss: 0.005918


In [40]:
# ============================================
# STEP 10.4 — Benign Reconstruction Error
# ============================================

reconstruction_model.eval()

with torch.no_grad():

    benign_reconstructed = reconstruction_model(
        benign_rep_tensor.to(DEVICE)
    ).cpu().numpy()

benign_mae = np.mean(
    np.abs(
        benign_reconstructed -
        benign_representations
    ),
    axis=1
)

print("Benign reconstruction error:")
print("Mean:", benign_mae.mean())
print("Median:", np.median(benign_mae))
print("Std:", benign_mae.std())
print("Max:", benign_mae.max())

Benign reconstruction error:
Mean: 0.06773039
Median: 0.051612705
Std: 0.16403739
Max: 6.932562


In [41]:
# ============================================
# STEP 10.5 — Reconstruction Train/Validation Split
# ============================================

from sklearn.model_selection import train_test_split

benign_rep_train, benign_rep_val = train_test_split(
    benign_representations,
    test_size=0.20,
    random_state=SEED
)

print("Reconstruction training samples:", benign_rep_train.shape)
print("Reconstruction validation samples:", benign_rep_val.shape)

Reconstruction training samples: (34145, 32)
Reconstruction validation samples: (8537, 32)


In [42]:
# ============================================
# STEP 10.6 — Reconstruction DataLoaders
# ============================================

benign_rep_train_tensor = torch.tensor(
    benign_rep_train,
    dtype=torch.float32
)

benign_rep_val_tensor = torch.tensor(
    benign_rep_val,
    dtype=torch.float32
)

reconstruction_train_dataset = TensorDataset(
    benign_rep_train_tensor,
    benign_rep_train_tensor
)

reconstruction_val_dataset = TensorDataset(
    benign_rep_val_tensor,
    benign_rep_val_tensor
)

reconstruction_train_loader = DataLoader(
    reconstruction_train_dataset,
    batch_size=256,
    shuffle=True
)

reconstruction_val_loader = DataLoader(
    reconstruction_val_dataset,
    batch_size=256,
    shuffle=False
)

In [43]:
# ============================================
# STEP 10.7 — Reconstruction Training
# ============================================

reconstruction_model = ReconstructionModel(
    input_dimension=32
).to(DEVICE)

reconstruction_optimizer = torch.optim.Adam(
    reconstruction_model.parameters(),
    lr=0.001
)

reconstruction_criterion = nn.MSELoss()

RECONSTRUCTION_EPOCHS = 30

best_val_loss = float("inf")
best_state = None

for epoch in range(RECONSTRUCTION_EPOCHS):

    # ----- Training -----
    reconstruction_model.train()

    train_loss = 0.0

    for representations, targets in reconstruction_train_loader:

        representations = representations.to(DEVICE)
        targets = targets.to(DEVICE)

        reconstruction_optimizer.zero_grad()

        reconstructed = reconstruction_model(representations)

        loss = reconstruction_criterion(
            reconstructed,
            targets
        )

        loss.backward()
        reconstruction_optimizer.step()

        train_loss += loss.item() * representations.size(0)

    train_loss /= len(reconstruction_train_loader.dataset)

    # ----- Validation -----
    reconstruction_model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for representations, targets in reconstruction_val_loader:

            representations = representations.to(DEVICE)
            targets = targets.to(DEVICE)

            reconstructed = reconstruction_model(representations)

            loss = reconstruction_criterion(
                reconstructed,
                targets
            )

            val_loss += loss.item() * representations.size(0)

    val_loss /= len(reconstruction_val_loader.dataset)

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_state = {
            key: value.detach().cpu().clone()
            for key, value in reconstruction_model.state_dict().items()
        }

    if (epoch + 1) % 5 == 0 or epoch == 0:

        print(
            f"Epoch {epoch + 1:02d}/{RECONSTRUCTION_EPOCHS} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

# Restore best model
reconstruction_model.load_state_dict(best_state)

print("\nBest validation loss:", best_val_loss)

Epoch 01/30 | Train Loss: 6.218567 | Val Loss: 0.982694
Epoch 05/30 | Train Loss: 0.003506 | Val Loss: 0.003239
Epoch 10/30 | Train Loss: 0.002977 | Val Loss: 0.003063
Epoch 15/30 | Train Loss: 0.001672 | Val Loss: 0.001522
Epoch 20/30 | Train Loss: 0.000782 | Val Loss: 0.000684
Epoch 25/30 | Train Loss: 0.000455 | Val Loss: 0.000341
Epoch 30/30 | Train Loss: 0.000503 | Val Loss: 0.000300

Best validation loss: 0.0003004936913143152


In [45]:
[name for name in globals() if "benign" in name.lower() or "reconstruction" in name.lower()]

['correct_benign_mask',
 'X_benign_correct',
 'benign_representations',
 'benign_rep_tensor',
 'reconstruction_dataset',
 'reconstruction_loader',
 'ReconstructionModel',
 'reconstruction_model',
 'reconstruction_optimizer',
 'reconstruction_criterion',
 'RECONSTRUCTION_EPOCHS',
 'benign_reconstructed',
 'benign_mae',
 'benign_rep_train',
 'benign_rep_val',
 'benign_rep_train_tensor',
 'benign_rep_val_tensor',
 'reconstruction_train_dataset',
 'reconstruction_val_dataset',
 'reconstruction_train_loader',
 'reconstruction_val_loader']

In [47]:
# ============================================
# STEP 11 — Benign Reconstruction Error
# ============================================

reconstruction_model.eval()

with torch.no_grad():
    benign_val_reconstructed = reconstruction_model(
        benign_rep_val_tensor.to(DEVICE)
    )

# Per-sample Mean Absolute Error (MAE)
benign_mae = torch.mean(
    torch.abs(
        benign_rep_val_tensor.to(DEVICE) - benign_val_reconstructed
    ),
    dim=1
).cpu().numpy()

print("Benign reconstruction error statistics:")
print(f"Mean   : {benign_mae.mean():.6f}")
print(f"Median : {np.median(benign_mae):.6f}")
print(f"Std    : {benign_mae.std():.6f}")
print(f"Min    : {benign_mae.min():.6f}")
print(f"Max    : {benign_mae.max():.6f}")

print("\nPercentiles:")
for percentile in [90, 95, 99, 99.5, 99.9]:
    print(
        f"{percentile}% : "
        f"{np.percentile(benign_mae, percentile):.6f}"
    )

Benign reconstruction error statistics:
Mean   : 0.007100
Median : 0.002521
Std    : 0.010074
Min    : 0.002009
Max    : 0.385340

Percentiles:
90% : 0.015706
95% : 0.020598
99% : 0.043366
99.5% : 0.055443
99.9% : 0.116553


In [52]:
# ============================================
# STEP 12A — READ Metric Functions
# ============================================

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import entropy as scipy_entropy


def calculate_medae(original_data, reconstructed_data):
    """
    Per-sample Median Absolute Error.
    """
    absolute_error = np.abs(original_data - reconstructed_data)
    return np.median(absolute_error, axis=1)


def calculate_mc_dropout_uncertainty(
    model,
    X,
    n_passes=20
):
    """
    Estimate:
    - Aleatoric uncertainty
    - Epistemic uncertainty
    - Predictive entropy

    Uses MC Dropout during inference.
    """

    model.train()  # activate Dropout

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    ).to(DEVICE)

    probability_samples = []

    with torch.no_grad():

        for _ in range(n_passes):

            logits = model(X_tensor)

            # Binary classification probability
            probabilities = torch.sigmoid(logits).squeeze(1)

            probability_samples.append(
                probabilities.cpu().numpy()
            )

    probability_samples = np.stack(
        probability_samples,
        axis=0
    )

    # Mean prediction across MC passes
    mean_probability = probability_samples.mean(axis=0)

    # ------------------------------------------------
    # Epistemic uncertainty
    # Variance of predictions across MC passes
    # ------------------------------------------------
    epistemic_uncertainty = probability_samples.var(
        axis=0
    )

    # ------------------------------------------------
    # Aleatoric uncertainty
    # Bernoulli predictive variance
    # ------------------------------------------------
    aleatoric_uncertainty = (
        mean_probability *
        (1.0 - mean_probability)
    )

    # ------------------------------------------------
    # Predictive entropy
    # ------------------------------------------------
    probability_matrix = np.column_stack([
        1.0 - mean_probability,
        mean_probability
    ])

    predictive_entropy = scipy_entropy(
        probability_matrix.T + 1e-12,
        axis=0
    )

    model.eval()

    return (
        aleatoric_uncertainty,
        epistemic_uncertainty,
        predictive_entropy
    )

In [53]:
# ============================================
# STEP 12B — READ Metrics for Clean Benign Data
# ============================================

# Use the held-out benign reconstruction validation samples
X_benign_read = benign_rep_val_tensor.cpu().numpy()

# Reconstruct benign samples
reconstruction_model.eval()

with torch.no_grad():
    X_benign_reconstructed = reconstruction_model(
        benign_rep_val_tensor.to(DEVICE)
    ).cpu().numpy()

# ------------------------------------------------
# 1. MedAE
# ------------------------------------------------
benign_medae = calculate_medae(
    X_benign_read,
    X_benign_reconstructed
)

# ------------------------------------------------
# 2. Aleatoric uncertainty
# 3. Epistemic uncertainty
# 4. Predictive entropy
# ------------------------------------------------
(
    benign_aleatoric,
    benign_epistemic,
    benign_entropy
) = calculate_mc_dropout_uncertainty(
    ids_model,
    X_benign_read,
    n_passes=20
)

# ------------------------------------------------
# Display statistics
# ------------------------------------------------

print("READ metrics — Clean Benign Samples")
print("=" * 50)

print(f"Number of samples: {len(X_benign_read)}")

print("\nMedAE:")
print(f"  Mean   : {benign_medae.mean():.6f}")
print(f"  Median : {np.median(benign_medae):.6f}")
print(f"  Std    : {benign_medae.std():.6f}")

print("\nAleatoric Uncertainty:")
print(f"  Mean   : {benign_aleatoric.mean():.6f}")
print(f"  Median : {np.median(benign_aleatoric):.6f}")
print(f"  Std    : {benign_aleatoric.std():.6f}")

print("\nEpistemic Uncertainty:")
print(f"  Mean   : {benign_epistemic.mean():.6f}")
print(f"  Median : {np.median(benign_epistemic):.6f}")
print(f"  Std    : {benign_epistemic.std():.6f}")

print("\nPredictive Entropy:")
print(f"  Mean   : {benign_entropy.mean():.6f}")
print(f"  Median : {np.median(benign_entropy):.6f}")
print(f"  Std    : {benign_entropy.std():.6f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (8537x32 and 79x128)